# 02. Raise

**Tier:** Intermediate. **Why you'd use this:** A booking form rejects a negative seat count. The helper should not return a fake number. It should stop the caller with a clear `ValueError`, and if a lower conversion failed first, keep that cause attached.

**After this lesson you can:** raise `ValueError` for a bad argument; chain causes with `raise ... from`; tell a bug you hide from a bug you explain.
About 25 minutes. **Uses:** [try/except](01-try-except-lesson.ipynb)


## Quick review

1. Why catch `ValueError` instead of bare `except`?
2. Is `KeyboardInterrupt` an `Exception`?

<details><summary>Answers</summary>

Narrow catches leave real bugs loud. `KeyboardInterrupt` is a `BaseException`, not an `Exception`.
</details>


## The idea

`raise` builds an exception and starts unwinding. Use it when the caller passed something your function refuses to handle. Returning a quiet stand-in would hide the broken booking. `raise NewError(...) from old` chains the old exception as the cause, so the traceback shows both layers instead of pretending the wrapper was the only failure.

<details><summary>Explain more</summary>

`raise ValueError("empty")` sets the message. Catching that error with `as err` lets you read `str(err)`. Chaining is for wrappers: you translate a low-level failure into a domain error without erasing what went wrong underneath. Skip the chain only when the low-level detail would mislead.
</details>


## See it work

Reject a negative seat count, and wrap a bad text conversion.


In [1]:
def seats(text):
    # 1. Convert, then validate.
    try:
        n = int(text)
    except ValueError as err:
        # 2. Keep the original failure as the cause.
        raise ValueError("seats must be an integer") from err
    if n < 0:
        raise ValueError("negative")
    return n


print(seats("3"))

3


## Check yourself

Answer these before you run the cell.

1. What does `seats("-1")` raise?
2. After the chained raise, what is `__cause__`?


In [2]:
def seats(text):
    try:
        n = int(text)
    except ValueError as err:
        raise ValueError("seats must be an integer") from err
    if n < 0:
        raise ValueError("negative")
    return n


try:
    seats("-1")
except ValueError as err:
    print(type(err).__name__, str(err))

try:
    seats("x")
except ValueError as err:
    print(str(err))
    print(type(err.__cause__).__name__)

ValueError negative
seats must be an integer
ValueError


<details><summary>Answers</summary>

`"-1"` raises `ValueError: negative`. `"x"` raises the wrapper message, and `__cause__` is the original `ValueError` from `int`.
</details>


## Why it works this way

`raise` does not return. It transfers control to the nearest matching handler, building a traceback of frames as it goes. Returning `None` for a negative seat count would force every caller to remember a magic value. Raising makes the failure loud at the boundary you chose. Deep dive: [Memory management](../../internals/11-memory/01-refcount-lesson.ipynb).

`raise new from old` sets `__cause__` on the new exception. The traceback printer shows the cause first, then the wrapper. That chain is the design for layered APIs: lower code raises what it knows, upper code renames the failure for its callers, and nobody has to guess which error was first. `raise new from None` is the other deliberate choice: it hides the cause when the low-level detail would only confuse.

The cost is allocating the exception and walking frames. Raise for contract violations, not for ordinary false results. A function that raises on bad input is documenting a rule the type system is not checking. Callers that care catch that type. Callers that do not let it surface.

Chaining also helps tests. You can assert on the wrapper message your API promises, and still assert that `__cause__` is a `ValueError` when the input was not an integer. Without `from`, that second assert has nothing to hold.

Prefer one clear exception type per failure mode at your boundary. `ValueError` for bad seats is enough. Inventing a new class can wait until callers need to catch your errors without also catching every other `ValueError` in the program.


In [3]:
def seats(text):
    try:
        n = int(text)
    except ValueError as err:
        raise ValueError("seats must be an integer") from err
    if n < 0:
        raise ValueError("negative")
    return n


try:
    seats("x")
except ValueError as err:
    print(err.__cause__ is not None)
    print(str(err.__cause__))

True
invalid literal for int() with base 10: 'x'


## When to use it

Raise when the caller broke your contract. Return a stand-in when bad input is normal and local, as in the previous lesson. Use `raise ... from` when you replace one exception with another. Use `from None` only when the cause would confuse more than it helps. Don't catch, log, and ignore a cause you just wrapped.


## Common mistake

Catching a conversion error and raising a new one without `from`. The traceback loses the original message, so you cannot see whether the text was empty or letters.


In [4]:
def seats_wrong(text):
    try:
        n = int(text)
    except ValueError:
        raise ValueError("seats must be an integer")
    return n


try:
    seats_wrong("x")
except ValueError as err:
    print(str(err))
    print(err.__cause__)

seats must be an integer
None


Chain with `from` so `__cause__` keeps the first error.


In [5]:
def seats(text):
    try:
        n = int(text)
    except ValueError as err:
        raise ValueError("seats must be an integer") from err
    return n


try:
    seats("x")
except ValueError as err:
    print(str(err))
    print(str(err.__cause__))

seats must be an integer
invalid literal for int() with base 10: 'x'


## Try it

Change `seats` so `0` is also rejected as `"empty"`. Should that use the same `ValueError` type, and why is the message the part callers should branch on carefully?


## Recap

- `raise` stops the happy path with an exception object.
- `raise new from old` keeps the cause in `__cause__`.
- Raise for broken contracts. Return stand-ins for expected junk.

**Next:** [practice](02-raise-practice.ipynb)
